# RFM Quant — Backtesting e investigação quantitativa

**Input → Modelo → Output → Estratégia → Ordens simuladas.** O objetivo é testar hipóteses históricas sem inventar rentabilidade nem executar ordens reais.

## 1. Gerar dados reais de histórico

A partir da raiz do repositório, executar no terminal:

```bash
pip install -r requirements-research.txt
python -m research.export_research --symbol BTC-USDT-SWAP --bar 15m --bars 2000
```

A exportação usa o endpoint público OKX e o gerador de candidatos experimental. Se existirem falhas no histórico, interrompe o teste.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from research.quant_pipeline import prepare_dataset, walk_forward, FEATURES
root = Path("research_outputs/BTC-USDT-SWAP/15m")
assert (root/"candles.csv").exists(), "Primeiro executa python -m research.export_research."

## 2. Inspecionar input, sinais e ordens simuladas

In [ ]:
candles = pd.read_csv(root/"candles.csv")
events = pd.read_csv(root/"events.csv")
trades = pd.read_csv(root/"trades.csv")
print("Velas:",len(candles),"Candidatos:",len(events),"Operações fechadas:",len(trades))
display(trades.head())

## 3. Construir features com informação disponível no momento do sinal

O resultado em R só é conhecido na saída da operação. Sinais sem execução não recebem artificialmente o rótulo de perda.

In [ ]:
if len(trades):
    dataset = prepare_dataset(candles,events,trades)
    display(dataset[["signal_ts","exit_ts","r_multiple","target_win"]+FEATURES].head())
else:
    dataset = pd.DataFrame()
    print("Sem operações fechadas; não é possível treinar XGBoost.")

## 4. XGBoost com walk-forward e purge

Os folds de treino usam apenas operações cuja saída ocorreu antes do primeiro sinal de teste. Comparamos o Brier score do XGBoost com um baseline simples. Sem dados suficientes, não são produzidas métricas artificiais.

In [ ]:
if len(dataset):
    report = walk_forward(dataset,min_train=80,test_size=25,n_splits=4)
    print("Estado:",report["status"])
    display(pd.DataFrame(report["folds"]))
    print("Brier XGBoost:",report.get("mean_xgboost_brier"))
    print("Brier baseline:",report.get("mean_baseline_brier"))
else:
    print("Dados insuficientes.")

## 5. Conclusões válidas

Brier inferior ao baseline não demonstra lucro. Os sinais continuam a ser gerados pela hipótese provisória de pivôs, não pela RFM completamente especificada. Faltam histórico longo, holdout intocado, custos de funding, validação por regimes e confirmação objetiva dos swings.